In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnableBranch, RunnableLambda

In [2]:
load_dotenv()
parser = StrOutputParser()

In [4]:
pip install grandalf

Note: you may need to restart the kernel to use updated packages.


Simple Chain

In [6]:
load_dotenv()
prompt = PromptTemplate(
    template="Generate 5 interesting facts about {topic}",
    input_variables=["topic"]
)
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
chain = prompt | model | parser
results = chain.invoke({
    "topic" : "Cricket"
})
chain.get_graph().print_ascii()
print(results)


1. Cricket is one of the oldest team sports in the world, with records of the game dating back to the 16th century. It is believed to have originated in England.

2. The longest cricket match in history lasted for 14 days, played between England and South Africa in 1939. The match was called off due to the outbreak of World War II.

3. The highest individual score in a Test match is held by Brian Lara, who scored 400 not out for the West Indies against England in 2004.

4. Cricket is known as a gentleman's game, with players expected to adhere to a strict code of conduct known as the "Spirit of Cricket." This includes fair play, respect for opponents, and upholding the integrity of the game.

5. The Cricket World Cup is one of the most-watched sporting events in the world, attracting millions of viewers and fans from around the globe. The tournament is held every four years and features teams from various countries competing for the prestigious title.
     +-------------+       
     |

Sequential Chain

In [3]:
prompt1 = PromptTemplate(
    template="Generate a detailed report about {topic}",
    input_variables=["topic"]
)
prompt2 = PromptTemplate(
    template="Generate 5 point summary from the following text : {text}",   
    input_variables=["text"]
)
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()
chain = prompt1| model | parser | prompt2| model | parser
chain.get_graph().print_ascii() 
results = chain.invoke({
    "topic" : "Unemployment in Pakistan"
})
print(results)

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

Parallel Chain

In [6]:
model1 = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
model2 = ChatOpenAI(model="gpt-4", temperature=0.5)

prompt1 = PromptTemplate(
    template= "Generate Short and simple notes from the following text : {text}",
    input_variables=["text"]
)
prompt2 = PromptTemplate(

    template= "Generate 5 short questions from the following text: {text}",
    input_variables=["text"]
)

prompt3 = PromptTemplate(
    template="Merge the provided notes and quiz into a single document:\n notes :  {notes} \n quiz : {quiz}",
    input_variables=["notes", "quiz"]
)


#parrallel chain
parrallel_chain = RunnableParallel({
    "notes" : prompt1 | model1 | parser,
    "quiz" : prompt2 | model2 | parser
})
merge_chain = prompt3 | model1 | parser
chain = parrallel_chain | merge_chain
text = """
The Industrial Revolution, which began in the late 18th century, was a period of significant technological and social change. It marked the transition from agrarian economies to industrialized ones, leading to increased production and urbanization. Key inventions during this time included the steam engine, spinning jenny, and power loom. The revolution also had profound effects on society, including changes in labor practices, the rise of the middle class, and shifts in population distribution. However, it also brought about challenges such as poor working conditions and environmental pollution."""

results = chain.invoke({
    "text": text
})
chain.get_graph().print_ascii() 

print(results)  


            +---------------------------+            
            | Parallel<notes,quiz>Input |            
            +---------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOpenAI |                    | ChatOpenAI |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

Conditional Chain

In [9]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
model= ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
class Feedback(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the feedback, either positive or negative.")

parser2 = PydanticOutputParser(pydantic_object=Feedback)
prompt1 = PromptTemplate(
    template="Classify the sentiment of the following feedback text into positive or negative: \n{feedback} \n {format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser2.get_format_instructions()}
)
classifier_chain = prompt1 | model | parser2
feedbacks =  "The product quality is terrible and I am very dissatisfied with my purchase."
prompt2= PromptTemplate(
    template="Write an appropriate response to the following positive feedback: \n{feedback}",
    input_variables=["feedback"],
)
prompt3= PromptTemplate(
    template="Write an appropriate response to the following negative feedback: \n{feedback}",
    input_variables=["feedback"],
)
# classification_results = classifier_chain.invoke({"feedback": feedbacks}).sentiment
# print("Sentiment Classification Result: ", classification_results)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == "positive", prompt2 | model | parser),
    (lambda x:x.sentiment == "negative", prompt3 | model | parser),
    (RunnableLambda(lambda x: "Neutral feedback, no response needed."))
)

chain = classifier_chain | branch_chain
print(chain.invoke({"feedback": feedbacks}))
chain.get_graph().print_ascii()

I'm sorry to hear that you had a negative experience. We always strive to provide the best service possible and we value your feedback. Please let us know how we can improve and make things right for you. Thank you for bringing this to our attention.
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *        